# Task 1: Data Cleaning & Preprocessing
Titanic dataset — missing values, categorical encoding, scaling and outlier handling.

In [ ]:
import urllib.request
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

DATA_URL = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
Path('data').mkdir(exist_ok=True)
Path('outputs').mkdir(exist_ok=True)
if not Path('data/titanic.csv').exists():
    urllib.request.urlretrieve(DATA_URL, 'data/titanic.csv')
df = pd.read_csv('data/titanic.csv')
print('Shape:', df.shape)
display(df.head())
display(df.dtypes)
display(df.isnull().sum())

## 1. Remove unnecessary columns and handle missing values

In [ ]:
df = df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'], errors='ignore')
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Fare'] = df['Fare'].fillna(df['Fare'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
print(df.isnull().sum())

## 2. Visualize and remove outliers using the IQR method

In [ ]:
for col in ['Age', 'Fare']:
    plt.figure(figsize=(6, 3))
    sns.boxplot(x=df[col])
    plt.title(f'{col} before outlier removal')
    plt.show()

before = len(df)
for col in ['Age', 'Fare']:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    df = df[df[col].between(lower, upper)]
print('Rows removed:', before - len(df))

## 3. Encode categorical features and standardize numerical features

In [ ]:
df = pd.get_dummies(df, columns=['Sex', 'Embarked'], drop_first=True, dtype=int)
scale_cols = ['Age', 'Fare', 'SibSp', 'Parch']
scaler = StandardScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])
display(df.head())
print('Final shape:', df.shape)
print('Missing values:', df.isnull().sum().sum())
df.to_csv('outputs/titanic_cleaned.csv', index=False)
print('Saved: outputs/titanic_cleaned.csv')